# Sentence Transformers

This notebook covers Sentence Transformers, a powerful library for generating high-quality sentence embeddings. Unlike word-level embeddings like Word2Vec or GloVe, sentence transformers produce dense vectors that capture the meaning of entire sentences or paragraphs.

Sentence Transformers are based on transformer models like BERT and are fine-tuned specifically for semantic similarity tasks.

In [ ]:
# Import required libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Libraries imported successfully.")

## 1. Understanding Sentence Transformers

Sentence Transformers are a modification of BERT/RoBERTa that are specifically trained for producing sentence-level embeddings. They use a siamese network structure during training to learn semantically meaningful sentence representations.

**Key advantages:**
- Produce fixed-length vectors for any sentence length
- Capture semantic meaning at the sentence level
- Excellent for semantic similarity, clustering, and retrieval
- Pre-trained on large datasets with 1 billion+ training pairs

**Popular models:**
- `all-MiniLM-L6-v2`: Fast, good quality, 384 dimensions
- `all-mpnet-base-v2`: Best quality, 768 dimensions
- `paraphrase-multilingual-MiniLM-L12-v2`: Multi-language support

In [ ]:
# Load a pre-trained sentence transformer model
# Using a smaller, faster model for demonstration
model = SentenceTransformer('all-MiniLM-L6-v2')

print("=== Sentence Transformer Model ===")
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 2. Encoding Sentences

Let's encode some sample sentences and examine the embeddings.

In [ ]:
# Sample sentences
sentences = [
    "The cat sat on the mat.",
    "A feline is resting on the rug.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "The weather is pleasant today.",
    "It's a beautiful day outside.",
]

# Encode sentences to get embeddings
embeddings = model.encode(sentences)

print("=== Sentence Embeddings ===")
print(f"Number of sentences: {len(sentences)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"\nFirst sentence: '{sentences[0]}'")
print(f"Embedding (first 10 values): {embeddings[0][:10]}")

In [ ]:
# Show that semantically similar sentences have similar embeddings
print("=== Semantic Similarity ===\n")

# Calculate pairwise similarities
similarity_matrix = cosine_similarity(embeddings)

print("Similarity Matrix:")
print("-" * 80)
for i in range(len(sentences)):
    row = ", ".join([f"{s:.2f}" for s in similarity_matrix[i]])
    print(f"{i}: {row}")

print("\nNote: Sentences 0-1 and 2-3 are semantically similar,")
print("and should have higher similarity scores.")

## 3. Finding Similar Sentences

Sentence Transformers excel at finding semantically similar sentences, which is useful for information retrieval, clustering, and duplicate detection.

In [ ]:
# Find most similar sentences
def find_similar_sentences(query, sentences, embeddings, model, top_k=5):
    """Find the most similar sentences to a query."""
    # Encode the query
    query_embedding = model.encode([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Get top K most similar
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append((sentences[idx], similarities[idx]))
    
    return results

# Test with a query
query = "A dog is sitting on the floor."
print(f"=== Query: '{query}' ===\n")

similar = find_similar_sentences(query, sentences, embeddings, model)
print("Most similar sentences:")
for i, (sent, score) in enumerate(similar, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Sentence: {sent}\n")

In [ ]:
# Another example
query = "AI and neural networks."
print(f"=== Query: '{query}' ===\n")

similar = find_similar_sentences(query, sentences, embeddings, model)
print("Most similar sentences:")
for i, (sent, score) in enumerate(similar, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Sentence: {sent}\n")

## 4. Semantic Search

Sentence Transformers are excellent for semantic search, where the meaning of the query matters more than exact keyword matching.

In [ ]:
# Create a document corpus for semantic search
documents = [
    "Python is a high-level programming language.",
    "Java is a popular object-oriented programming language.",
    "Machine learning enables computers to learn from data.",
    "Deep learning is a subset of machine learning using neural networks.",
    "Natural language processing deals with understanding human language.",
    "Computer vision is about teaching computers to interpret images.",
    "Reinforcement learning is about agents learning from rewards.",
    "Data science involves extracting insights from data.",
    "Neural networks are inspired by the human brain.",
    "Transformers have revolutionised natural language processing.",
]

# Encode all documents
doc_embeddings = model.encode(documents)

print("=== Semantic Search ===")
print(f"Indexed {len(documents)} documents")
print(f"Embedding dimension: {doc_embeddings.shape}")

In [ ]:
# Perform semantic search
def semantic_search(query, documents, doc_embeddings, model, top_k=3):
    """Perform semantic search on a document corpus."""
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'document': documents[idx],
            'score': similarities[idx],
            'index': idx
        })
    
    return results

# Test searches
test_queries = [
    "programming languages",
    "artificial intelligence and learning",
    "image recognition",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = semantic_search(query, documents, doc_embeddings, model)
    for r in results:
        print(f"Score: {r['score']:.4f} - {r['document']}")

## 5. Sentence Clustering

Sentence embeddings can be used for clustering similar sentences together.

In [ ]:
from sklearn.cluster import KMeans

# Create a diverse set of sentences
clustering_sentences = [
    # Technology cluster
    "Python is a programming language.",
    "Java is used for software development.",
    "C++ is a systems programming language.",
    
    # Science cluster
    "The Earth orbits around the Sun.",
    "Water boils at 100 degrees Celsius.",
    "Light travels at 300,000 km per second.",
    
    # Food cluster
    "Apples are delicious fruits.",
    "Coffee contains caffeine.",
    "Bread is made from flour.",
    
    # Sports cluster
    "Football is a popular sport.",
    "Cricket is played with a bat and ball.",
    "Tennis uses rackets to hit balls.",
]

# Encode sentences
cluster_embeddings = model.encode(clustering_sentences)

# Perform K-means clustering
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_embeddings)

print("=== Sentence Clustering ===")
for i in range(n_clusters):
    print(f"\nCluster {i}:")
    for j, sent in enumerate(clustering_sentences):
        if clusters[j] == i:
            print(f"  - {sent}")

## 6. Different Sentence Transformer Models

Let's explore some other available models and their characteristics.

In [ ]:
print("=== Available Sentence Transformer Models ===")
print("""
Popular models from the Sentence Transformers library:

1. all-MiniLM-L6-v2 (384 dimensions)
   - Fast and efficient
   - Good quality for most tasks
   - Recommended for production

2. all-mpnet-base-v2 (768 dimensions)
   - Best quality embeddings
   - Slower than MiniLM
   - Use when accuracy is critical

3. paraphrase-multilingual-MiniLM-L12-v2 (384 dimensions)
   - Supports 50+ languages
   - Good for cross-lingual tasks

4. all-roberta-large-v1 (1024 dimensions)
   - Large model, best quality
   - Slower inference
   - Use for research

To list all available models:
```python
from sentence_transformers import SentenceTransformer
print(SentenceTransformer('all-MiniLM-L6-v2').__class__.__name__)
```
""")

## 7. Batch Encoding

For large corpora, batch encoding is more efficient.

In [ ]:
# Create a larger corpus for batch processing
large_corpus = [
    f"Document {i}: This is a sample sentence for testing purposes."
    for i in range(100)
]

# Encode in batches
batch_embeddings = model.encode(large_corpus, batch_size=32, show_progress_bar=True)

print("=== Batch Encoding ===")
print(f"Encoded {len(large_corpus)} documents")
print(f"Output shape: {batch_embeddings.shape}")

## Summary

In this notebook, we covered Sentence Transformers:

1. **Understanding**: Learned about sentence-level embeddings vs word embeddings
2. **Encoding**: Encoded sentences to get dense vector representations
3. **Similarity**: Calculated semantic similarity between sentences
4. **Semantic Search**: Implemented semantic search over document collections
5. **Clustering**: Clustered similar sentences using K-means
6. **Models**: Explored different pre-trained sentence transformer models
7. **Batch Processing**: Learned about efficient batch encoding

Sentence Transformers are state-of-the-art for sentence-level semantic tasks. They provide high-quality embeddings that capture meaning better than simple averages of word embeddings, making them ideal for semantic search, clustering, and similarity tasks.